# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\medha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\medha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data\\HealthWellnessGuide.txt', 'data\\MentalHealthGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [15]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [20]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Property 'headlines' already exists in node '2af1ab'. Skipping!
Property 'headlines' already exists in node '58f0ca'. Skipping!


Applying HeadlineSplitter:   0%|          | 0/9 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

...

Applying CustomNodeFilter:   0%|          | 0/14 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/30 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '2af1ab'. Skipping!
Property 'summary_embedding' already exists in node '58f0ca'. Skipping!
Property 'entities' already exists in node '6e9396'. Skipping!
Property 'themes' already exists in node 'd1b751'. Skipping!
Property 'themes' already exists in node 'e49813'. Skipping!
Property 'themes' already exists in node '984b12'. Skipping!
Property 'entities' already exists in node '984b12'. Skipping!
Property 'themes' already exists in node '8a94db'. Skipping!
Property 'themes' already exists in node '2af86a'. Skipping!
Property 'themes' already exists in node '78f391'. Skipping!
Property 'themes' already exists in node '6e9396'. Skipping!
Property 'entities' already exists in node '2af86a'. Skipping!
Property 'entities' already exists in node 'e49813'. Skipping!
Property 'entities' already exists in node 'd1b751'. Skipping!
Property 'entities' already exists in node '78f391'. Skipping!
Property 'entities' already exists in node '8a94db'

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 16, relationships: 37)

We can save and load our knowledge graphs as follows.

In [21]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 16, relationships: 37)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [22]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [23]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
*Your answer here*

Finally, we can use our `TestSetGenerator` to generate our testset!

In [24]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,How can CBT-I help in managing insomia accordi...,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,Cognitive Behavioral Therapy for Insomnia (CBT...,single_hop_specifc_query_synthesizer
1,"As a Holistic Wellness Coach, how can maintain...",[13: The Science of Habit Formation Habits are...,Maintaining a healthy work-life balance is ess...,single_hop_specifc_query_synthesizer
2,what do i do on wednesday for exercise?,[The Personal Wellness Guide A Comprehensive R...,"On Wednesday, the recommended activity is to r...",single_hop_specifc_query_synthesizer
3,What World Health Organization say about menta...,[The Mental Health and Psychology Handbook A P...,"According to the World Health Organization, me...",single_hop_specifc_query_synthesizer
4,What is Mindfulness-Based Cognitive Therapy (M...,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Mindfulness-Based Cognitive Therapy (MBCT) com...,single_hop_specifc_query_synthesizer
5,How can understanding the science of habit for...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,"Understanding the science of habit formation, ...",multi_hop_abstract_query_synthesizer
6,How can nutriion and healthy eating be combine...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Nutrition and healthy eating involve consuming...,multi_hop_abstract_query_synthesizer
7,how exercise and physical activity help improv...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Exercise and physical activity play a crucial ...,multi_hop_abstract_query_synthesizer
8,how nutrition and diet from chapter 4 help men...,[<1-hop>\n\nWrite letters to or from your futu...,Nutrition and diet play a crucial role in ment...,multi_hop_specific_query_synthesizer
9,how chapter 12 mindfulness and meditation help...,[<1-hop>\n\nsocial interactions How to set and...,Chapter 12 explains that mindfulness and medit...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [25]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [26]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How do vitmains help in mental well-being?,[The Personal Wellness Guide A Comprehensive R...,Vitamins are organic compounds needed in small...,single_hop_specifc_query_synthesizer
1,"As a Mental Wellness Advocate, could you expla...",[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Chamomile is mentioned as one of the herbal te...,single_hop_specifc_query_synthesizer
2,What is Agoraphobia and how it affect peoples ...,[The Mental Health and Psychology Handbook A P...,Agoraphobia is a type of anxiety disorder char...,single_hop_specifc_query_synthesizer
3,What are the core principles and common techni...,[Chapter 4: Cognitive Behavioral Therapy (CBT)...,Cognitive Behavioral Therapy (CBT) is a widely...,single_hop_specifc_query_synthesizer
4,how digital mental health challenges like soci...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Digital mental health challenges such as socia...,multi_hop_abstract_query_synthesizer
5,How do digital mental health challenges such a...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,"Digital mental health challenges, including so...",multi_hop_abstract_query_synthesizer
6,How do the recommendations for exercise and mo...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,The Personal Wellness Guide emphasizes that re...,multi_hop_abstract_query_synthesizer
7,How do mindfulness and meditation practices co...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Mindfulness and meditation practices contribut...,multi_hop_abstract_query_synthesizer
8,How do the strategies for managing digital men...,[<1-hop>\n\nwith others. Types of boundaries: ...,Chapter 16 outlines strategies for managing di...,multi_hop_specific_query_synthesizer
9,How do the strategies for managing digital men...,[<1-hop>\n\nwith others. Types of boundaries: ...,Chapter 16 outlines strategies for managing di...,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
*Your answer here*

---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [ ]:
### YOUR CODE HERE ###

# Define a custom query distribution with different weights
# Generate a new test set and compare with the default


We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [27]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [28]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [30]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [32]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [33]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [34]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [35]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [36]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [37]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [38]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [39]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [40]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`:
> - `labeled_helpfulness_evaluator`:
> - `dopeness_evaluator`:

## LangSmith Evaluation

In [41]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'best-pull-27' at:
https://smith.langchain.com/o/28fa6697-9440-4a5b-b7c7-e89547a40996/datasets/46b18adb-fe97-426d-b8d3-383c25bd79ce/compare?selectedSessions=f0b840b8-c80c-4dad-9548-c8c1c38b54fb




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,how journaling and exercise both help mental h...,Journaling and exercise both help mental healt...,None,Journaling helps mental health by allowing ind...,True,True,True,7.696204,95a8033f-8773-4db4-8ffb-3ee820e30cdf,019c72d4-af4e-74e2-a2c4-23c357a15ea4
1,how journaling and exercise help mental health...,Based on the provided context:\n\nJournaling h...,None,Journaling helps mental health by allowing ind...,True,True,False,4.931233,d9545a8f-0038-4b94-8872-e1aaf51e4bd5,019c72d5-0704-7082-8873-88fa90ff6e39
2,How do the strategies for managing digital men...,The strategies for managing digital mental hea...,None,Chapter 16 outlines strategies for managing di...,True,True,True,3.786968,6cac7232-8dd6-454d-851c-c932bfcb0e16,019c72d5-46c0-7882-af91-4a7104e0e2ff
3,How do the strategies for managing digital men...,The strategies for managing digital mental hea...,None,Chapter 16 outlines strategies for managing di...,True,True,True,4.563759,22255ce6-a4d1-4d62-94c5-1d3af88c0ff5,019c72d5-8a52-77a1-bf3f-44ee1d74d849
4,How do mindfulness and meditation practices co...,Based on the provided context:\n\nMindfulness ...,None,Mindfulness and meditation practices contribut...,True,True,True,7.111214,7bdb6886-213f-4436-9547-9d22f5c97fa9,019c72d5-cb53-7c41-a5de-e7dd9962bc6e
5,How do the recommendations for exercise and mo...,The recommendations for exercise and movement ...,None,The Personal Wellness Guide emphasizes that re...,True,True,True,4.934412,16eb6214-f63d-481d-a8d6-0f9568b3f6c0,019c72d6-221f-7962-b8ca-69b1a4c4da1f
6,How do digital mental health challenges such a...,Digital mental health challenges related to so...,None,"Digital mental health challenges, including so...",True,True,False,4.105045,ae9db21d-bf18-4fbe-852d-bcd446b5932d,019c72d6-6178-72f1-aed1-3ed634d64b10
7,how digital mental health challenges like soci...,Digital mental health challenges such as socia...,None,Digital mental health challenges such as socia...,True,True,True,4.426232,e51e780c-cae9-4a99-afef-2d2a2b093231,019c72d6-a5d8-7c23-b37d-081ec1b189ef
8,What are the core principles and common techni...,The core principles of Cognitive Behavioral Th...,None,Cognitive Behavioral Therapy (CBT) is a widely...,True,True,False,3.583927,37c16574-cc64-42db-a991-53b86433e1ae,019c72d6-fc1e-7e32-b6c1-3aabbbe45063
9,What is Agoraphobia and how it affect peoples ...,Agoraphobia is a fear of situations where esca...,None,Agoraphobia is a type of anxiety disorder char...,True,True,False,1.908291,202b9928-fe07-4198-90c7-d2d7c1569ffc,019c72d7-3e55-7e60-887a-169ecc0bfccf


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [42]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [43]:
rag_documents = docs

In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
*Your answer here*

In [45]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
*Your answer here*

In [46]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [47]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [48]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [49]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

'Alright, ready to turbocharge your sleep game? Here’s your VIP pass to dreamland mastery straight from the sleep sages:\n\n1. **Lock in a sleep schedule like a boss**: Go to bed and rise at the same time every day, weekends included. Your body LOVES rhythm—it’s the ultimate sleep DJ spinning consistent beats.\n\n2. **Ritualize your wind-down**: Think gentle stretching, diving into a good book, or soaking in a warm bath. This bedtime routine whispers to your brain, “Chill time is here.”\n\n3. **Optimize your sleep fortress**:\n   - Keep your bedroom a chilly chill zone at 65-68°F (18-20°C)—think cool, not frozen.\n   - Blank out the world with blackout curtains or a sleep mask. Darkness is your sleep homie.\n   - Silence the noise beasts using earplugs or white noise machines, turning your space into a zen cocoon.\n   - Upgrade your mattress and pillows because comfort is queen or king here.\n\n4. **Screen detox pre-sleep**: Power down devices 1-2 hours before hitting the sack. Blue li

Finally, we can evaluate the new chain on the same test set!

In [ ]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'abandoned-liquid-36' at:
https://smith.langchain.com/o/340cd80b-3296-5752-9a9e-58582118073a/datasets/79646c5e-82fe-4bbc-85e9-567b67cd20a9/compare?selectedSessions=ab0e9212-346f-4459-890b-d8fede4ebb90




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How does Cognitive Behavioral Therapy (CBT) fo...,"Oh, this is where science meets slick sleep sw...",None,Cognitive Behavioral Therapy for Insomnia (CBT...,True,True,True,6.233257,6532273f-6b9f-427b-b9ab-7bd3b03433bb,019c49a6-a2d5-7701-84db-2c39606df922
1,"How can Cognitive Behavioral Therapy (CBT), in...","Alright, buckle up for a mind-melding deep div...",None,"Cognitive Behavioral Therapy (CBT), particular...",True,True,True,8.946787,37917403-6be6-496e-ac59-b19c7db30426,019c49a6-f5fa-7121-ab9a-98723078502b
2,H0w c4n I bUild a he4lthy m0rn1ng r0utine (cha...,"Yo, let’s craft that legendary morning routine...",None,To build a healthy morning routine that improv...,True,True,True,5.125188,d68152dc-ef9e-4a74-81ce-7c7cb301468a,019c49a7-5128-7863-9c0f-00e49330da3b
3,how chapter 7 and 17 connect about sleep and h...,"Alright, let’s crank up the dopeness on how Ch...",None,"chapter 7 talks about sleep and recovery, expl...",True,True,True,6.317486,505279e2-422f-4f5d-865a-fd4cf423e79d,019c49a7-a7b3-7ce2-a1bf-bd089b843876
4,How can I improve my emotional intelligence an...,"Alright, let’s crank this up to eleven on the ...",None,To improve emotional intelligence and manage c...,True,True,True,10.900828,c65e7315-6d08-4abd-b256-46460dc3130a,019c49a7-ecd7-7001-a74d-4ce6bf8c60a7
5,How can improving face-to-face interactions an...,"Oh yeah, let’s ride this wave of mental health...",None,Improving face-to-face interactions by engagin...,True,True,True,3.910628,a3102956-bd1c-4bce-89e5-cd197f1ba3d0,019c49a8-3dd9-7780-b63a-91703e2383f2
6,How can incorporating mindfulness and social c...,"Alright, buckle up for this mental health mast...",None,Incorporating mindfulness and social connectio...,True,True,True,5.030634,b5194163-70c0-4062-9ed5-8da997bb64ab,019c49a8-8937-7921-97e1-d8f224168372
7,how can exercise for common problems like lowe...,"Yo, here’s the sick synergy between those move...",None,The wellness guide explains that gentle exerci...,True,True,True,4.505386,6602cc2a-535c-4921-a62f-0d739b80b910,019c49a8-b659-7ff2-8be0-0cdeb6b0d50d
8,How does the World Health Organization define ...,"Alright, buckle up for some straight-up mental...",None,"According to the World Health Organization, me...",True,True,True,2.954818,02d2ea31-3c58-4b8d-a254-bfbc0fb4423b,019c49a8-fb15-7811-bb5f-8aaf6d843ebc
9,What information does Chapter 18 cover regardi...,"Oh snap, Chapter 18 is the immune system’s ult...",None,Chapter 18 discusses strategies to boost immun...,True,True,True,4.718449,206be997-a104-4644-a1e1-7a47dd9e828c,019c49a9-2a87-73a2-9edc-6905004e07f1


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:
*Your answer here*

---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores